## Conventions & Palettes
- Mapping-only visuals; non-map analysis lives in `high_paying_jobs_data_visualization.ipynb`.
- Consistent palettes:
  - Money/Income: Blues
  - Concentration (LQ): Purples
  - Education premium: Greens
- All maps save to `Images/` at 300 DPI with tight layout.

# High-Paying Jobs in the US — Mapping (Choropleths)

Scope: This notebook focuses on geospatial analysis only (state-level choropleths and maps). All non-map charts (bars, violins, heatmaps, regressions) live in `high_paying_jobs_data_visualization.ipynb` to avoid duplication.

Conventions:
- Income/wage scales use a sequential blue palette
- Counts/headcounts use a neutral grey palette
- Diverging values (e.g., z-scores) use a coolwarm palette
- Images are saved to `Images/` with descriptive filenames

See `high_paying_jobs_data_visualization.ipynb` for complementary non-map figures and detailed narratives.

In [ ]:
# =============================================================================
# 1. SETUP AND CONFIGURATION
# =============================================================================
# Core imports
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# Optional geospatial libs
try:
    import geopandas as gpd
    HAS_GEOPANDAS = True
except ImportError:
    gpd = None
    HAS_GEOPANDAS = False

try:
    import shapefile  # pyshp
    HAS_PYSHP = True
except ImportError:
    shapefile = None
    HAS_PYSHP = False

# Paths and constants
DF_PATH = './Data/cleaned_high_pay_data.csv'
SHAPEFILE_PATH = './us_state/us_state.shp'

PALETTE = {
    'money_seq': 'Blues',            # wages/income (sequential)
    'count_seq': 'Greys',            # counts/headcounts (sequential)
    'gender': ['#1f77b4', '#ff7f0e'],# fixed gender colors
    'diverge': 'coolwarm',           # for standardized metrics
    'lq_seq': 'Purples',             # for LQ
    'premium_seq': 'Greens'          # for education premium
}

# Plot style
plt.style.use('default')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'white'

# Ensure output directory exists
os.makedirs('Images', exist_ok=True)

print('✅ Environment ready for geographic analysis')
print(' - GeoPandas available:', HAS_GEOPANDAS)
print(' - pyshp available:', HAS_PYSHP)

In [ ]:
# =============================================================================
# 2. DATA LOADING (single source of truth)
# =============================================================================
print('📊 Loading high-paying jobs data...')
try:
    high_pay_data = pd.read_csv(DF_PATH)
    print(f'✅ Data loaded: {len(high_pay_data):,} records')
except FileNotFoundError:
    print(f'❌ Error: File not found at {DF_PATH}')
    high_pay_data = pd.DataFrame()

# Load geographic data (with fallback if GeoPandas missing)
if HAS_GEOPANDAS:
    try:
        us_states = gpd.read_file(SHAPEFILE_PATH)
        us_states = ensure_stusps_gdf(us_states)
        print("🗺️  Shapefile loaded and 'STUSPS' confirmed")
    except Exception as e:
        print(f'❌ Error loading shapefile at {SHAPEFILE_PATH}: {e}')
        us_states = None
else:
    print('🗺️  GeoPandas not available; pyshp fallback will be used for static PNGs')
    us_states = None

> Note: All saved plot filenames are unique across this notebook to avoid accidental overwrites. The helper functions above use explicit, distinct names per figure.

In [ ]:
def StatesPlot(df, column_to_plot, cmap='viridis', label_color='black', label_size=6,
               title='United States Map', filename='us_map.png', min_value=None, max_value=None,
               edge_color='black', edge_linewidth=0.5, colorbar_label=None):
    """
    Numeric choropleth using GeoPandas; respects vmin/vmax and writes to Images/.
    """
    if column_to_plot not in df.columns:
        raise ValueError(f"'{column_to_plot}' column not found in the DataFrame.")
    tmp = df.dropna(subset=[column_to_plot]).copy()
    vmin = min_value if min_value is not None else float(tmp[column_to_plot].min())
    vmax = max_value if max_value is not None else float(tmp[column_to_plot].max())
    fig, ax = plt.subplots(figsize=(15, 10), dpi=300)
    ax.set_axis_off()
    tmp.plot(column=column_to_plot, ax=ax, alpha=0.7, cmap=cmap, linewidth=edge_linewidth, edgecolor=edge_color,
             vmin=vmin, vmax=vmax)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04, aspect=10)
    cbar.set_label(colorbar_label or column_to_plot, fontsize=12)
    has_stusps = 'STUSPS' in tmp.columns
    if has_stusps:
        for _, row in tmp.iterrows():
            if row.get('geometry') is not None:
                try:
                    c = row['geometry'].centroid
                    ax.text(c.x, c.y, row['STUSPS'], fontsize=label_size, ha='center', color=label_color)
                except Exception:
                    pass
    ax.set_title(title, fontsize=18, pad=15)
    os.makedirs('Images', exist_ok=True)
    plt.savefig(os.path.join('Images', filename), bbox_inches='tight', dpi=300)
    print(f"Map successfully saved to Images/{filename}")
    plt.show()

def Education_State(df, column, title, filename, cmap='tab20'):
    """Categorical choropleth for dominant education level per state."""
    fig, ax = plt.subplots(1, 1, figsize=(18, 12))
    df = df.dropna(subset=[column])
    df.plot(column=column, ax=ax, cmap=cmap, legend=True, linewidth=0.5, edgecolor='black',
            legend_kwds={'title': 'Education Level by State'})
    if 'STUSPS' in df.columns:
        for _, row in df.iterrows():
            c = row['geometry'].centroid
            ax.text(c.x, c.y, row['STUSPS'], fontsize=12, ha='center', color='black')
    plt.title(title, fontsize=18)
    plt.axis('off')
    if ax.get_legend() is not None:
        ax.get_legend().set_bbox_to_anchor((1.05, 0.5))
    os.makedirs('Images', exist_ok=True)
    plt.savefig(os.path.join('Images', filename), bbox_inches='tight', dpi=300)
    print(f"Map successfully saved to Images/{filename}")
    plt.show()

def plot_gender_distribution(df):
    """Overlay-style gender distribution map (male/female %)."""
    fig, ax = plt.subplots(1, 1, figsize=(20, 14))
    df.plot(column='Male_Percentage', ax=ax, cmap='Blues', linewidth=0.5, edgecolor='black', alpha=0.7)
    df.plot(column='Female_Percentage', ax=ax, cmap='Reds', linewidth=0.5, edgecolor='black', alpha=0.5)
    for _, row in df.iterrows():
        ax.text(row['geometry'].centroid.x, row['geometry'].centroid.y,
                row.get('STUSPS', row.get('State Abbreviation', '')),
                ha='center', fontsize=8, color='black')
    blue_patch = plt.Line2D([0], [0], color='blue', lw=4, label='Male Percentage')
    red_patch = plt.Line2D([0], [0], color='red', lw=4, label='Female Percentage')
    plt.legend(handles=[blue_patch, red_patch], loc='lower left', fontsize=12)
    plt.title('Distribution of Male and Female Percentages in High-Paying States ($100K+)', fontsize=18)
    ax.set_aspect('equal')
    plt.axis('off')
    plt.savefig('Images/Gender_Distribution_Overlay.png', bbox_inches='tight', pad_inches=0.1, dpi=300)
    print('Map successfully saved to Images/Gender_Distribution_Overlay.png')
    plt.show()

In [ ]:
# =============================================================================
# 3. DATA PREPARATION
# =============================================================================
# Aggregate high-paying jobs and total jobs by state
print("📊 Aggregating high-paying jobs and total jobs by state...")
try:
    job_data = (
        high_pay_data.groupby('State Abbreviation')
        .agg(
            High_Paying_Jobs=('Annual Income', 'size'),
            Total_Jobs=('Employment', 'size'),
            Location_Quotient=('Location Quotient', 'mean')  # Average LQ for better representation
        )
        .reset_index()
    )
    print(f"✅ Aggregation completed: {len(job_data)} states analyzed")
except KeyError as e:
    print(f"❌ Error during aggregation: {e}")
    job_data = pd.DataFrame()



Note: This notebook intentionally avoids non-map analyses (bars, violins, heatmaps, regressions). Use `high_paying_jobs_data_visualization.ipynb` for those to prevent duplication.

Note: This notebook intentionally avoids non-map analyses (bars, violins, heatmaps, regressions). Use `data_viz.ipynb` for those to prevent duplication.

In [ ]:
# Calculate the average annual income by state
A_income_state = (
    high_pay_data.groupby(['State Abbreviation'])['Annual Income']
.mean()
.reset_index()
.round(2)
)

# If GeoPandas or the shapefile isn't available, render a static PNG with pyshp fallback
if not HAS_GEOPANDAS or us_states is None:
    print("⚠️ GeoPandas or shapefile unavailable. Using pyshp fallback to export a PNG choropleth.")
    os.makedirs('Images', exist_ok=True)
    plot_income_choropleth_pyshp(
        income_df=A_income_state,
        shp_path=SHAPEFILE_PATH,
        out_png='Images/Average_Highest_Income_State.png',
        title='Average Highest Income by State',
        edge_color='black', edge_linewidth=0.3, label_color='black', label_size=8,
        exclude_states=('AK', 'HI')
)
    geo_income_data = None
else:
    us_states = ensure_stusps_gdf(us_states)
    geo_income_data = A_income_state.merge(us_states, left_on='State Abbreviation', right_on='STUSPS', how='inner')
    geo_income_data = gpd.GeoDataFrame(geo_income_data, geometry=geo_income_data.geometry)
    if 'STUSPS' in geo_income_data.columns:
        geo_income_data = geo_income_data.loc[~geo_income_data['STUSPS'].isin(['AK', 'HI'])]
        print("✅ Excluded AK and HI for contiguous US visualization.")
    print("✅ Geo merge complete:", len(geo_income_data), "states ready for mapping.")
    StatesPlot(
        df=geo_income_data,
        column_to_plot='Annual Income',
        cmap=PALETTE['money_seq'],
        title='Average Highest Income by State',
        filename='Average_Highest_Income_State.png',
        label_color='black',
        label_size=8,
        edge_color='black',
        edge_linewidth=0.5,
        colorbar_label='Average annual income (USD)'
)

In [ ]:
# Q1 - Interpretation (auto-generated)
from IPython.display import Markdown, display

_state_income = high_pay_data.groupby('State Abbreviation')['Annual Income'].mean()
_top_state = _state_income.idxmax(); _top_val = _state_income.max()
_bottom_state = _state_income.idxmin(); _bottom_val = _state_income.min()
_national_avg = high_pay_data['Annual Income'].mean()

_md = f"""
### Interpretation (Q1)
- Highest average income: {_top_state} (${_top_val:,.0f})
- Lowest average income: {_bottom_state} (${_bottom_val:,.0f})
- National average: ${_national_avg:,.0f}

The map highlights strong cross-state disparities in high-paying job income levels.
"""
display(Markdown(_md))

In [ ]:
# Q2 - Step 1: Aggregate High-Paying Job Data by State
job_data_q2 = (
    high_pay_data.groupby('State Abbreviation')
.agg(
        High_Paying_Jobs=('Annual Income', 'size'),
        Total_Jobs=('Employment', 'size'),
        Location_Quotient=('Location Quotient', 'mean')
)
.reset_index()
)

# Q2 - Step 2: Plot High-Paying Jobs Distribution (Counts)
if HAS_GEOPANDAS and us_states is not None:
    us_states_q2 = ensure_stusps_gdf(us_states.copy())
    job_data_geo = pd.merge(us_states_q2, job_data_q2, left_on='STUSPS', right_on='State Abbreviation', how='left')
    job_data_geo = gpd.GeoDataFrame(job_data_geo, geometry=job_data_geo.geometry)
    if 'STUSPS' in job_data_geo.columns:
        job_data_geo = job_data_geo.loc[~job_data_geo['STUSPS'].isin(['AK', 'HI'])]
    StatesPlot(
        df=job_data_geo,
        column_to_plot='High_Paying_Jobs',
        cmap='YlGnBu',
        label_color='black',
        label_size=8,
        title='High-Paying Jobs Distribution Across the United States',
        filename='High_Paying_Jobs_Distribution.png',
        edge_color='black',
        colorbar_label='High-paying jobs (count)'
)
else:
    print("⚠️ GeoPandas unavailable; using shapefile fallback for High_Paying_Jobs map.")
    plot_df = job_data_q2.copy()
    if plot_df['High_Paying_Jobs'].isna().any():
        plot_df['High_Paying_Jobs'] = plot_df['High_Paying_Jobs'].fillna(0)
    plot_income_choropleth_pyshp(
        income_df=plot_df,
        shp_path=SHAPEFILE_PATH,
        out_png='Images/High_Paying_Jobs_Distribution.png',
        title='High-Paying Jobs Distribution Across the United States',
        edge_color='black', edge_linewidth=0.3, label_color='black', label_size=8,
        exclude_states=('AK', 'HI'),
        value_col='High_Paying_Jobs',
        colorbar_label='High-Paying Jobs (count)',
        cmap='YlGnBu'
)

In [ ]:
# Q2 (Counts) - Interpretation (auto-generated)
from IPython.display import Markdown, display

# Compute counts robustly from the source data
_counts = high_pay_data.groupby('State Abbreviation').size()
_max_state = _counts.idxmax(); _max_val = int(_counts.max())
_min_state = _counts.idxmin(); _min_val = int(_counts.min())
_total = int(_counts.sum())

_md2 = f"""
### Interpretation (Q2 - Counts)
- Highest count of high-paying roles: {_max_state} ({_max_val:,})
- Lowest count of high-paying roles: {_min_state} ({_min_val:,})
- Total high-paying roles (all states): {_total:,}

States with larger economies and populations dominate the job count distribution.
"""
display(Markdown(_md2))

### Interpretation (Counts): Where are high-paying roles concentrated?
- Darker states indicate higher absolute counts of $100K+ roles.
- Large states with sizable labor markets often dominate on counts (e.g., CA, TX, NY).
- Use the concentration (LQ) view next to understand relative intensity beyond size effects.

In [ ]:
# Q2 - Step 2: Plot Location Quotient (LQ) Distribution
if HAS_GEOPANDAS and us_states is not None:
    try:
        lq_geo = job_data_geo.copy()
    except NameError:
        us_states_q2b = ensure_stusps_gdf(us_states.copy())
        lq_geo = pd.merge(us_states_q2b, job_data_q2, left_on='STUSPS', right_on='State Abbreviation', how='left')
        lq_geo = gpd.GeoDataFrame(lq_geo, geometry=lq_geo.geometry)
    if 'STUSPS' in lq_geo.columns:
        lq_geo = lq_geo.loc[~lq_geo['STUSPS'].isin(['AK', 'HI'])]
    lq_min = max(0.5, float(np.nanmin(lq_geo['Location_Quotient']))) if 'Location_Quotient' in lq_geo.columns else None
    lq_max = min(2.0, float(np.nanmax(lq_geo['Location_Quotient']))) if 'Location_Quotient' in lq_geo.columns else None
    StatesPlot(
        df=lq_geo,
        column_to_plot='Location_Quotient',
        cmap='YlGnBu',
        label_color='black',
        label_size=8,
        title='Location Quotient (LQ) for High-Paying Jobs',
        filename='High_Paying_Jobs_LQ_Distribution.png',
        edge_color='black',
        min_value=lq_min,
        max_value=lq_max,
        colorbar_label='LQ (relative concentration)'
)
else:
    print("⚠️ GeoPandas unavailable; using shapefile fallback for Location Quotient map.")
    plot_df = job_data_q2.copy()
    plot_df['Location_Quotient'] = plot_df['Location_Quotient'].fillna(0)
    plot_income_choropleth_pyshp(
        income_df=plot_df,
        shp_path=SHAPEFILE_PATH,
        out_png='Images/High_Paying_Jobs_LQ_Distribution.png',
        title='Location Quotient (LQ) for High-Paying Jobs',
        edge_color='black', edge_linewidth=0.3, label_color='black', label_size=8,
        exclude_states=('AK', 'HI'),
        value_col='Location_Quotient',
        colorbar_label='LQ (relative concentration)',
        cmap='YlGnBu'
)

In [ ]:
# Q2 (LQ) - Interpretation (auto-generated)
from IPython.display import Markdown, display

# Compute average LQ per state robustly
if 'Location_Quotient' in high_pay_data.columns:
    _lq = high_pay_data.groupby('State Abbreviation')['Location_Quotient'].mean()
else:
    _lq = job_data.groupby('State Abbreviation')['Location_Quotient'].mean()

_lq_top = _lq.idxmax(); _lq_top_val = _lq.max()
_lq_bottom = _lq.idxmin(); _lq_bottom_val = _lq.min()

_md_lq = f"""
### Interpretation (Q2 - Location Quotient)
- Highest concentration (LQ): {_lq_top} ({_lq_top_val:.2f})
- Lowest concentration (LQ): {_lq_bottom} ({_lq_bottom_val:.2f})

Values above 1.0 indicate states where high-paying jobs are more concentrated than the national average.
"""
display(Markdown(_md_lq))

### Interpretation (Concentration): Which states stand out after size-adjustment?
- Higher LQ (>1) suggests a greater-than-average share of high-paying roles relative to national distribution.
- Smaller states with specialized industries can rank high on LQ even if absolute counts are modest.
- Use counts + LQ together: big markets vs. specialized concentrations.

## Question 2: Which states have the highest number and the highest concentration of high-paying jobs ($100K+ annual income)?

We’ll visualize two aspects:
- Count of high-paying jobs by state (High_Paying_Jobs).
- Concentration via Location Quotient (LQ), which contextualizes concentration relative to national averages.

### Interpretation: High-Paying Jobs Concentration
- The states shaded darker indicate higher counts of roles earning $100K+ annually.
- Large economies and tech/finance hubs typically top the list (e.g., CA, NY, TX, MA).
- Consider normalizing by population or total employment for rate-based comparisons (to avoid size bias).
- Outliers or unexpectedly low/high states could reflect industry mix or sampling within the dataset.

## Question 3: What is the dominant education level for high-paying jobs ($100K+) across different states, and how does this reflect regional job market demands?

> We identify, for each state, the education level that appears most often among high-paying jobs and then map it to reveal regional patterns.

In [ ]:
# Q3 - Data Preparation: Dominant education level per state for $100K+ jobs
dominant_education = (
    high_pay_data.groupby(['State Abbreviation', 'Education Level'])
.size()
.reset_index(name='Count')
)
dominant_education = dominant_education.sort_values(
    ['State Abbreviation', 'Count'], ascending=[True, False]
).drop_duplicates(subset='State Abbreviation', keep='first')

# Q3 - Visualization
if HAS_GEOPANDAS and us_states is not None:
    us_states_q3 = ensure_stusps_gdf(us_states.copy())
    geo_dominant_education = pd.merge(
        us_states_q3, dominant_education, left_on='STUSPS', right_on='State Abbreviation', how='left'
)
    geo_dominant_education = gpd.GeoDataFrame(geo_dominant_education, geometry=geo_dominant_education.geometry)
    if 'STUSPS' in geo_dominant_education.columns:
        geo_dominant_education = geo_dominant_education.loc[~geo_dominant_education['STUSPS'].isin(['AK', 'HI'])]
    Education_State(
        geo_dominant_education, 'Education Level',
        'Dominant Education Level in High-Paying States ($100K+)',
        'Dominant_Education_By_State.png',
        cmap='tab20'
)
else:
    print("⚠️ GeoPandas unavailable; using shapefile fallback for dominant education map.")
    cat_df = dominant_education[['State Abbreviation','Education Level']].copy()
    plot_categorical_choropleth_pyshp(
        cat_df=cat_df,
        shp_path=SHAPEFILE_PATH,
        out_png='Images/Dominant_Education_By_State.png',
        title='Dominant Education Level in High-Paying States ($100K+)',
        edge_color='black', edge_linewidth=0.3, label_color='black', label_size=8,
        exclude_states=('AK', 'HI'),
        category_col='Education Level',
        cmap='tab20',
        legend_title='Dominant education level'
)

In [ ]:
# Q3 - Interpretation (auto-generated)
from IPython.display import Markdown, display

_edu_mode = high_pay_data.groupby('State Abbreviation')['Education Level'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown')
_edu_counts = _edu_mode.value_counts()
_top_edu = _edu_counts.idxmax(); _top_edu_n = int(_edu_counts.max())

_md3 = f"""
### Interpretation (Q3)
- Most common dominant education level across states: {_top_edu} ({_top_edu_n} states)

States show distinct skilling patterns; align talent strategies with the modal education requirement per state.
"""
display(Markdown(_md3))

### Interpretation (Dominant Education): Regional demand signals
- States with bachelor’s or graduate-level dominance likely reflect knowledge-intensive sectors (tech, finance, healthcare).
- States showing associate or some college dominance may indicate strong demand for skilled technical roles and certifications.
- Cross-compare this map with counts and LQ to identify both large markets and specialized education-driven niches.

## Question 4: What is the gender distribution in high-paying jobs ($100K+) across different states?

> We compute male/female shares among high-paying jobs per state, then map them to spot regional patterns.

In [ ]:
# Q4 - Data Preparation: Gender distribution per state for $100K+ jobs
gender_counts = high_pay_data.groupby(['State Abbreviation', 'Gender']).size().unstack(fill_value=0)
gender_counts['Total'] = gender_counts.sum(axis=1)
gender_counts['Male_Percentage'] = np.where(gender_counts['Total']>0, (gender_counts.get('Male', 0) / gender_counts['Total']) * 100, np.nan)
gender_counts['Female_Percentage'] = np.where(gender_counts['Total']>0, (gender_counts.get('Female', 0) / gender_counts['Total']) * 100, np.nan)
gender_counts.reset_index(inplace=True)

# Q4 - Visualization
if HAS_GEOPANDAS and us_states is not None:
    us_states_q4 = ensure_stusps_gdf(us_states.copy())
    geo_gender_df = us_states_q4.merge(gender_counts, left_on='STUSPS', right_on='State Abbreviation', how='left')
    if 'STUSPS' in geo_gender_df.columns:
        geo_gender_df = geo_gender_df.loc[~geo_gender_df['STUSPS'].isin(['AK', 'HI'])]
    geo_gender_df = gpd.GeoDataFrame(geo_gender_df, geometry=geo_gender_df.geometry)
    geo_gender_df = geo_gender_df.dropna(how='any')
    plot_gender_distribution(geo_gender_df)
else:
    print("⚠️ GeoPandas unavailable; using shapefile fallback for gender distribution.")
    def fallback_plot_percentage(value_col, title, out_png, cmap):
        cat = gender_counts[['State Abbreviation', value_col]].copy()
        plot_income_choropleth_pyshp(
            income_df=cat.rename(columns={value_col: 'value'}).rename(columns={'value': value_col}),
            shp_path=SHAPEFILE_PATH,
            out_png=out_png,
            title=title,
            edge_color='black', edge_linewidth=0.3, label_color='black', label_size=8,
            exclude_states=('AK', 'HI'),
            value_col=value_col,
            colorbar_label=f"{value_col.replace('_',' ')} (%)",
            cmap=cmap
)
    fallback_plot_percentage('Male_Percentage', 'Male Share of High-Paying Jobs ($100K+)', 'Images/Male_Percentage_State.png', 'Blues')
    fallback_plot_percentage('Female_Percentage', 'Female Share of High-Paying Jobs ($100K+)', 'Images/Female_Percentage_State.png', 'Reds')

In [ ]:
# Q4 - Interpretation (auto-generated)
from IPython.display import Markdown, display

_gender_state = high_pay_data.pivot_table(index='State Abbreviation', columns='Gender', values='Annual Income', aggfunc='size', fill_value=0)
_gender_state['Total'] = _gender_state.sum(axis=1)
_gender_state['Female_%'] = _gender_state.get('Female', 0) / _gender_state['Total'] * 100
_gender_state['Male_%'] = _gender_state.get('Male', 0) / _gender_state['Total'] * 100

_avg_female = _gender_state['Female_%'].mean()
_avg_male = _gender_state['Male_%'].mean()
_balanced_state = (_gender_state['Female_%'] - 50).abs().idxmin()
_balanced_gap = (_gender_state.loc[_balanced_state, 'Female_%'] - 50)

_md4 = f"""
### Interpretation (Q4)
- Average female share across states: {_avg_female:.1f}%
- Average male share across states: {_avg_male:.1f}%
- Most balanced state around parity (~50% female): {_balanced_state} ({50+_balanced_gap:.1f}% female)

Separate male/female maps help reveal regional disparities; a combined overlay offers a quick parity view.
"""
display(Markdown(_md4))

### Interpretation (Gender Distribution):
- Blue/red intensity indicates the share of male/female among high-paying roles by state.
- Compare both maps (or overlays) to spot regions with more balanced vs. skewed representation.
- Consider intersecting with industry mix by state to explain differences (e.g., tech vs. healthcare vs. energy).

## Question 5: How do regional patterns (Northeast, South, Midwest, West) compare in high-paying job opportunities and income levels?

> We assign each state to a U.S. Census-style region, compute regional aggregates, and visualize income and job distribution patterns.

In [ ]:
# Q5 - Regional Analysis: Data Preparation and Visualization
print("🗺️  Analyzing regional patterns across the United States...")

# Define US regions
us_regions = {
    'Northeast': ['CT', 'ME', 'MA', 'NH', 'NJ', 'NY', 'PA', 'RI', 'VT'],
    'South': ['AL', 'AR', 'DE', 'FL', 'GA', 'KY', 'LA', 'MD', 'MS', 'NC', 'OK', 'SC', 'TN', 'TX', 'VA', 'WV'],
    'Midwest': ['IL', 'IN', 'IA', 'KS', 'MI', 'MN', 'MO', 'NE', 'ND', 'OH', 'SD', 'WI'],
    'West': ['AK', 'AZ', 'CA', 'CO', 'HI', 'ID', 'MT', 'NV', 'NM', 'OR', 'UT', 'WA', 'WY']
}

# Create region mapping
state_to_region = {}
for region, states in us_regions.items():
    for state in states:
        state_to_region[state] = region

# Add region column to data
high_pay_data['Region'] = high_pay_data['State Abbreviation'].map(state_to_region)

# Regional analysis
regional_stats = high_pay_data.groupby('Region').agg({
    'Annual Income': ['mean', 'median', 'std', 'count'],
    'State Abbreviation': 'nunique',
    'Gender': lambda x: (x == 'Female').sum() / len(x) * 100,  # Female percentage
    'Education Level': lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'  # Dominant education
}).round(2)

# Flatten column names
regional_stats.columns = ['_'.join(col).strip() for col in regional_stats.columns]
regional_stats = regional_stats.rename(columns={
    'Annual Income_mean': 'Avg_Income',
    'Annual Income_median': 'Median_Income', 
    'Annual Income_std': 'Income_StdDev',
    'Annual Income_count': 'Total_Jobs',
    'State Abbreviation_nunique': 'Num_States',
    'Gender_<lambda>': 'Female_Percentage',
    'Education Level_<lambda>': 'Dominant_Education'
})

print("✅ Regional analysis data prepared")
print(f"📊 Regions analyzed: {list(regional_stats.index)}")
print(f"📈 Total jobs across regions: {regional_stats['Total_Jobs'].sum():,}")

# Display summary
print(f"\n🔍 Regional Overview:")
for region in regional_stats.index:
    stats = regional_stats.loc[region]
    print(f"   • {region}: ${stats['Avg_Income']:,.0f} avg, {stats['Total_Jobs']:,} jobs, {stats['Num_States']} states")

# Question 5 Visualization: Comprehensive Regional Analysis
print("🎨 Creating comprehensive regional pattern analysis...")

# Create 2x2 subplot for regional analysis
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 16), dpi=300)

# Professional color scheme for regions
region_colors = {
    'Northeast': '#FF6B6B',  # Red
    'South': '#4ECDC4',      # Teal  
    'Midwest': '#45B7D1',    # Blue
    'West': '#96CEB4'        # Green
}

# 1. Average Income by Region
regions = regional_stats.index
avg_incomes = regional_stats['Avg_Income']
colors1 = [region_colors[region] for region in regions]

bars1 = ax1.bar(regions, avg_incomes, color=colors1, alpha=0.8, edgecolor='black')
ax1.set_title('Average Income by Region\nHigh-Paying Jobs ($100K+)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Average Annual Income ($)', fontsize=12)
ax1.tick_params(axis='x', rotation=45)

# Add value labels
for bar, value in zip(bars1, avg_incomes):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(avg_incomes)*0.01,
            f'${value:,.0f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# 2. Job Distribution by Region (Pie Chart)
job_counts = regional_stats['Total_Jobs']
wedges, texts, autotexts = ax2.pie(job_counts.values, labels=job_counts.index, autopct='%1.1f%%',
                                  colors=[region_colors[region] for region in job_counts.index],
                                  startangle=90, textprops={'fontsize': 11})
ax2.set_title('Job Distribution by Region\nTotal High-Paying Positions', fontsize=14, fontweight='bold')

# Make percentage text bold
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

# 3. Regional Income vs Job Volume Scatter
ax3.scatter(regional_stats['Total_Jobs'], regional_stats['Avg_Income'], 
           c=[region_colors[region] for region in regional_stats.index],
           s=300, alpha=0.7, edgecolors='black', linewidth=2)

# Add region labels
for i, region in enumerate(regional_stats.index):
    ax3.annotate(region, 
                (regional_stats.iloc[i]['Total_Jobs'], regional_stats.iloc[i]['Avg_Income']),
                xytext=(10, 10), textcoords='offset points', 
                fontsize=11, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor=region_colors[region], alpha=0.7))

ax3.set_xlabel('Total High-Paying Jobs', fontsize=12)
ax3.set_ylabel('Average Income ($)', fontsize=12)
ax3.set_title('Regional Market Analysis\nIncome vs Job Volume', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

# 4. Regional Characteristics Comparison
characteristics = ['Avg_Income', 'Median_Income', 'Female_Percentage']
x = np.arange(len(characteristics))
width = 0.2

for i, region in enumerate(regions):
    values = [
        regional_stats.loc[region, 'Avg_Income'] / 1000,  # Convert to thousands for readability
        regional_stats.loc[region, 'Median_Income'] / 1000,
        regional_stats.loc[region, 'Female_Percentage']
    ]
    bars = ax4.bar(x + i*width, values, width, label=region, 
                  color=region_colors[region], alpha=0.8, edgecolor='black')

ax4.set_xlabel('Characteristics', fontsize=12)
ax4.set_ylabel('Values', fontsize=12)
ax4.set_title('Regional Characteristics Comparison\n(Income in $1000s, Gender %)', fontsize=14, fontweight='bold')
ax4.set_xticks(x + width * 1.5)
ax4.set_xticklabels(['Avg Income\n($1000s)', 'Median Income\n($1000s)', 'Female %'])
ax4.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax4.grid(True, alpha=0.3)

plt.suptitle('Question 5: Regional Patterns in High-Paying Jobs Analysis', 
             fontsize=18, fontweight='bold', y=0.98)
plt.tight_layout()

# Save the plot
os.makedirs("Images", exist_ok=True)
plt.savefig('Images/Regional_Patterns_Analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Question 5 regional analysis completed!")
print(f"📊 Key Insights:")
print(f"   • Highest Income Region: {regional_stats['Avg_Income'].idxmax()} (${regional_stats['Avg_Income'].max():,.0f})")
print(f"   • Largest Job Market: {regional_stats['Total_Jobs'].idxmax()} ({regional_stats['Total_Jobs'].max():,} jobs)")
print(f"   • Most Gender Balanced: {regional_stats['Female_Percentage'].idxmax()} ({regional_stats['Female_Percentage'].max():.1f}% female)")

# Create summary table
print(f"\n📋 Regional Summary Table:")
print(regional_stats[['Avg_Income', 'Total_Jobs', 'Num_States', 'Female_Percentage']].round(0))

In [ ]:
# Q5 - Interpretation (auto-generated)
from IPython.display import Markdown, display

top_income = regional_stats['Avg_Income'].idxmax()
top_income_val = regional_stats['Avg_Income'].max()

top_jobs = regional_stats['Total_Jobs'].idxmax()
top_jobs_val = regional_stats['Total_Jobs'].max()

top_female = regional_stats['Female_Percentage'].idxmax()
top_female_val = regional_stats['Female_Percentage'].max()

md = f"""
### Interpretation (Q5)
- Highest average income: {top_income} (${top_income_val:,.0f})
- Largest job market: {top_jobs} ({top_jobs_val:,} jobs)
- Most female representation: {top_female} ({top_female_val:.1f}%)

Overall takeaways:
- Income and job volume vary by region; see the scatter for trade-offs.
- Bar and pie charts confirm the relative strength of each region.
- Use this as a dashboard to prioritize regions by either pay or scale.
"""

display(Markdown(md))

---
## Question 6: What is the relationship between education level and income premiums across different states?
#### Data Preparation

In [ ]:
# Question 6: Education-Income Premium Analysis by State
print("🎓 Analyzing education level and income premium relationships...")

# National baseline by education
national_education_income = high_pay_data.groupby('Education Level')['Annual Income'].mean()
bachelor_baseline = national_education_income.get("Bachelor's degree", 0)
print(f"📊 National baseline (Bachelor's): ${bachelor_baseline:,.0f}")

# Compute premiums per state relative to Bachelor's within the same state
state_education_analysis = []
for state in high_pay_data['State Abbreviation'].dropna().unique():
    state_data = high_pay_data[high_pay_data['State Abbreviation'] == state]
    state_edu_income = state_data.groupby('Education Level')['Annual Income'].mean()
    bachelor_income = state_edu_income.get("Bachelor's degree", np.nan)
    if pd.notna(bachelor_income) and bachelor_income > 0:
        for edu_level, income in state_edu_income.items():
            premium_pct = (income - bachelor_income) / bachelor_income * 100
            premium_dollar = income - bachelor_income
            sample_size = int((state_data['Education Level'] == edu_level).sum())
            state_education_analysis.append({
                'State': state,
                'Education_Level': edu_level,
                'Income': float(income),
                'Premium_Percent': float(premium_pct),
                'Premium_Dollar': float(premium_dollar),
                'Sample_Size': sample_size
            })

edu_premium_df = pd.DataFrame(state_education_analysis)

# State-level metrics
state_edu_metrics = high_pay_data.groupby('State Abbreviation').agg({
    'Education Level': ['nunique', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'],
    'Annual Income': ['mean', 'std']
}).round(2)
state_edu_metrics.columns = ['_'.join(col).strip() for col in state_edu_metrics.columns]
state_edu_metrics = state_edu_metrics.rename(columns={
    'Education Level_nunique': 'Education_Diversity',
    'Education Level_<lambda>': 'Dominant_Education',
    'Annual Income_mean': 'State_Avg_Income',
    'Annual Income_std': 'Income_Variability'
})

advanced_levels = ["Master's degree", 'Professional degree', 'Doctoral degree']
advanced_premiums = (
    edu_premium_df[edu_premium_df['Education_Level'].isin(advanced_levels)]
.groupby('State')['Premium_Percent']
.mean()
.sort_values(ascending=False)
)

print("✅ Education-income premium analysis completed")
print(f"📈 Top states by advanced-degree premium: {advanced_premiums.head(3).round(1).to_dict() if not advanced_premiums.empty else {}}")

In [ ]:
# Q6 Visualization and Interpretation
os.makedirs('Images', exist_ok=True)

if edu_premium_df.empty:
    print("No premium data to visualize (missing Bachelor's baseline per state).")
else:
    state_summary = (
        edu_premium_df.groupby('State')
.agg(Avg_Premium_Percent=('Premium_Percent', 'mean'),
             Median_Premium_Percent=('Premium_Percent', 'median'),
             N=('Sample_Size', 'sum'))
.sort_values('Avg_Premium_Percent', ascending=False)
)
    top_adv = advanced_premiums.head(10) if not advanced_premiums.empty else pd.Series(dtype=float)

    fig, axes = plt.subplots(2, 2, figsize=(20, 16), dpi=300)

    # 1) Average premium percent by state (bar)
    ax = axes[0, 0]
    sns.barplot(x=state_summary.index, y=state_summary['Avg_Premium_Percent'], ax=ax, palette='viridis')
    ax.set_title("Average Premium (%) by State vs Bachelor's", fontsize=14, fontweight='bold')
    ax.set_ylabel('Average Premium (%)')
    ax.set_xlabel('State')
    ax.tick_params(axis='x', rotation=90)

    # 2) Top 10 states for advanced degrees premium (bar)
    ax = axes[0, 1]
    if not top_adv.empty:
        sns.barplot(x=top_adv.index, y=top_adv.values, ax=ax, palette='magma')
        ax.set_title('Top 10 States: Advanced Degree Premium (%)', fontsize=14, fontweight='bold')
        ax.set_ylabel('Avg Premium (%) for Advanced Degrees')
        ax.set_xlabel('State')
        ax.tick_params(axis='x', rotation=90)
    else:
        ax.text(0.5, 0.5, 'No advanced degree data', ha='center', va='center')
        ax.axis('off')

    # 3) Distribution of premiums by state (boxplot)
    ax = axes[1, 0]
    sns.boxplot(data=edu_premium_df, x='State', y='Premium_Percent', ax=ax, showfliers=False)
    ax.set_title('Distribution of Education Premiums by State', fontsize=14, fontweight='bold')
    ax.set_ylabel('Premium (%)')
    ax.set_xlabel('State')
    ax.tick_params(axis='x', rotation=90)

    # 4) Heatmap: Premium by education level and state
    ax = axes[1, 1]
    heat_df = (
        edu_premium_df.pivot_table(index='Education_Level', columns='State', values='Premium_Percent', aggfunc='mean')
.reindex(["Less than high school", "High school diploma or equivalent", "Some college, no degree", "Associate's degree", "Bachelor's degree", "Master's degree", "Professional degree", "Doctoral degree"], axis=0)
)
    sns.heatmap(heat_df, cmap='coolwarm', center=0, ax=ax)
    ax.set_title("Premium (%) by Education Level vs Bachelor's (State-wise)", fontsize=14, fontweight='bold')
    ax.set_xlabel('State')
    ax.set_ylabel('Education Level')

    plt.suptitle('Question 6: Education vs Income Premiums Across States', fontsize=18, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig('Images/Education_Income_Premiums_by_State.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Saved figure: Images/Education_Income_Premiums_by_State.png")

# Interpretation
if not edu_premium_df.empty:
    _state_avg = edu_premium_df.groupby('State')['Premium_Percent'].mean().sort_values(ascending=False)
    _top_state = _state_avg.index[0]; _top_val = _state_avg.iloc[0]
    _edu_avg = edu_premium_df.groupby('Education_Level')['Premium_Percent'].mean().sort_values(ascending=False)
    _top_edu = _edu_avg.index[0]; _top_edu_val = _edu_avg.iloc[0]
    _adv_head = advanced_premiums.head(1) if 'advanced_premiums' in globals() else None
    _adv_txt = '' if _adv_head is None or _adv_head.empty else f"Advanced degree premium leader: {_adv_head.index[0]} ({_adv_head.iloc[0]:.1f}%)."
    display(Markdown(f"""
### Interpretation (Q6)
- State with highest average premium (vs Bachelor's): {_top_state} ({_top_val:.1f}%)
- Education level with highest average premium (national): {_top_edu} ({_top_edu_val:.1f}%)
- {_adv_txt}

Premiums are calculated relative to the Bachelor's degree within each state. Positive values suggest higher returns for the education level compared to Bachelor's in that state.
"""))

---
## Question 7: How do industries/occupations distribute geographically in terms of high-paying opportunities?
#### Data Preparation & Visualization

In [ ]:
# Q7: Geographic Distribution of Industries/Occupations (High-Paying)
print("🏭 Analyzing geographic distribution of high-paying industries...")

# Determine occupation column flexibly
occ_candidates = ['Occupation_Title', 'Occupation Title', 'Occupation', 'occupation', 'OCCUPATION', 'Job Title', 'Job_Title']
occ_col = next((c for c in occ_candidates if c in high_pay_data.columns), None)
if occ_col is None:
    print("⚠️ No occupation column found in dataset. Skipping Q7 plots.")
else:
    top_occupations = high_pay_data[occ_col].value_counts().head(10)
    occupation_geography = {}
    for occupation in top_occupations.index:
        occ_data = high_pay_data[high_pay_data[occ_col] == occupation]
        state_dist = occ_data['State Abbreviation'].value_counts()
        total_jobs = len(occ_data)
        top_state = state_dist.index[0] if len(state_dist) > 0 else 'N/A'
        top_state_pct = (state_dist.iloc[0] / total_jobs * 100) if len(state_dist) > 0 else 0
        avg_income = occ_data['Annual Income'].mean()
        occupation_geography[occupation] = {
            'Total_Jobs': total_jobs,
            'Top_State': top_state,
            'Top_State_Percentage': top_state_pct,
            'Avg_Income': avg_income,
            'State_Distribution': state_dist.head(5).to_dict(),
            'Geographic_Spread': len(state_dist)
        }

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 16), dpi=300)

    # 1. Occupation Concentration by Top State
    occupations = list(occupation_geography.keys())[:8]
    top_state_pcts = [occupation_geography[occ]['Top_State_Percentage'] for occ in occupations]
    bars1 = ax1.barh(range(len(occupations)), top_state_pcts,
                      color=plt.cm.Blues(np.linspace(0.35, 0.95, len(occupations))),
                      alpha=0.9, edgecolor='white', linewidth=0.5)
    ax1.set_yticks(range(len(occupations)))
    ax1.set_yticklabels([occ.replace(' ', '\n') if len(occ) > 20 else occ for occ in occupations], fontsize=10)
    ax1.set_xlabel('Concentration in Top State (%)')
    ax1.set_title('Geographic Concentration by Occupation\n% Jobs in Dominant State', fontsize=14, fontweight='bold')
    for bar, value in zip(bars1, top_state_pcts):
        ax1.text(bar.get_width() + (max(top_state_pcts) if top_state_pcts else 0)*0.01,
                 bar.get_y() + bar.get_height()/2, f'{value:.1f}%', va='center', fontsize=10)
    ax1.grid(axis='x', color='#dddddd', linewidth=0.7, alpha=0.6)

    # 2. Geographic Spread vs Average Income
    spread_values = [occupation_geography[occ]['Geographic_Spread'] for occ in occupations]
    income_values = [occupation_geography[occ]['Avg_Income'] for occ in occupations]
    scatter = ax2.scatter(spread_values, income_values, c=top_state_pcts, s=150, alpha=0.75,
                          cmap='viridis', edgecolors='black', linewidths=0.5)
    ax2.set_xlabel('Geographic Spread (# States)')
    ax2.set_ylabel('Average Income ($)')
    ax2.set_title('Geographic Spread vs Income by Occupation', fontsize=14, fontweight='bold')
    for i, occ in enumerate(occupations):
        ax2.annotate(occ.split()[-1], (spread_values[i], income_values[i]), xytext=(5,5), textcoords='offset points', fontsize=9)
    cbar2 = plt.colorbar(scatter, ax=ax2)
    cbar2.set_label('Top State Concentration (%)')
    ax2.grid(True, axis='both', color='#eeeeee', linewidth=0.7, alpha=0.6)

    # 3. Regional Occupation Mix (Stacked)
    if 'Region' not in high_pay_data.columns:
        us_regions = {
            'Northeast': ['CT', 'ME', 'MA', 'NH', 'NJ', 'NY', 'PA', 'RI', 'VT'],
            'South': ['AL', 'AR', 'DE', 'FL', 'GA', 'KY', 'LA', 'MD', 'MS', 'NC', 'OK', 'SC', 'TN', 'TX', 'VA', 'WV'],
            'Midwest': ['IL', 'IN', 'IA', 'KS', 'MI', 'MN', 'MO', 'NE', 'ND', 'OH', 'SD', 'WI'],
            'West': ['AK', 'AZ', 'CA', 'CO', 'HI', 'ID', 'MT', 'NV', 'NM', 'OR', 'UT', 'WA', 'WY']
        }
        state_to_region = {s:r for r, lst in us_regions.items() for s in lst}
        high_pay_data['Region'] = high_pay_data['State Abbreviation'].map(state_to_region)

    top_5_occs = top_occupations.head(5).index
    region_data = []
    for region in [r for r in high_pay_data['Region'].dropna().unique()]:
        region_totals = high_pay_data[high_pay_data['Region'] == region][occ_col].value_counts()
        for occ in top_5_occs:
            count = region_totals.get(occ, 0)
            region_data.append({'Region': region, 'Occupation': occ, 'Count': count})
    region_occ_df = pd.DataFrame(region_data)
    pivot_data = region_occ_df.pivot(index='Region', columns='Occupation', values='Count').fillna(0)
    bottom = np.zeros(len(pivot_data.index))
    colors3 = plt.cm.Set2(np.linspace(0.2, 0.9, len(pivot_data.columns)))
    for i, occ in enumerate(pivot_data.columns):
        label = occ if len(occ) <= 20 else ' '.join(occ.split()[:2]) + '…'
        ax3.bar(pivot_data.index, pivot_data[occ], bottom=bottom, label=label, color=colors3[i], alpha=0.9, edgecolor='white', linewidth=0.5)
        bottom += pivot_data[occ]
    ax3.set_ylabel('Number of Jobs')
    ax3.set_title('Top Occupations by Region (High-Paying)')
    ax3.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9, frameon=False, title='Occupation')
    ax3.tick_params(axis='x', rotation=45)
    ax3.grid(axis='y', color='#dddddd', linewidth=0.7, alpha=0.6)

    # 4. State Specialization Index
    specialization_data = []
    for state in high_pay_data['State Abbreviation'].dropna().unique():
        state_total = len(high_pay_data[high_pay_data['State Abbreviation'] == state])
        if state_total >= 20:
            for occ in top_occupations.head(6).index:
                state_occ_count = len(high_pay_data[(high_pay_data['State Abbreviation'] == state) & (high_pay_data[occ_col] == occ)])
                if state_occ_count > 0:
                    national_occ_pct = len(high_pay_data[high_pay_data[occ_col] == occ]) / len(high_pay_data)
                    state_occ_pct = state_occ_count / state_total
                    specialization_score = (state_occ_pct / national_occ_pct) if national_occ_pct > 0 else 0
                    specialization_data.append({'State': state,'Occupation': occ,'Specialization_Score': specialization_score,'Job_Count': state_occ_count})
    spec_df = pd.DataFrame(specialization_data)
    top_specializations = spec_df.nlargest(15, 'Specialization_Score')
    bars4 = ax4.barh(range(len(top_specializations)), top_specializations['Specialization_Score'], color='#d62728', alpha=0.9, edgecolor='white', linewidth=0.5)
    ax4.set_yticks(range(len(top_specializations)))
    labels4 = [f"{row['State']}: {row['Occupation'].split()[-1]}" for _, row in top_specializations.iterrows()]
    ax4.set_yticklabels(labels4, fontsize=9)
    ax4.set_xlabel('Specialization Score (>1 = Above National Avg)')
    ax4.set_title('Top 15 State-Occupation Specializations')
    ax4.axvline(x=1, color='black', linestyle='--', alpha=0.7, label='National Average')
    ax4.legend(frameon=False)
    ax4.grid(axis='x', color='#dddddd', linewidth=0.7, alpha=0.6)

    plt.suptitle('Question 7: Geographic Distribution of High-Paying Industries/Occupations', fontsize=18, fontweight='bold', y=0.98)
    plt.tight_layout()
    os.makedirs('Images', exist_ok=True)
    plt.savefig('Images/Industry_Geographic_Distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Saved figure: Images/Industry_Geographic_Distribution.png")

---
## Question 8: How does job market size correlate with income premiums across states (market size vs premium analysis)?
#### Data Preparation & Visualization

In [ ]:
# Q8: Market Size vs Income Premium Analysis
print("📈 Analyzing relationship between market size and income premiums...")

# Compute market metrics
market_analysis = high_pay_data.groupby('State Abbreviation').agg({
    'Annual Income': ['mean', 'median', 'std', 'count'],
    'Employment': 'sum'
}).round(2)

# Flatten columns
market_analysis.columns = ['_'.join(col).strip() for col in market_analysis.columns]
market_analysis = market_analysis.rename(columns={
    'Annual Income_mean': 'Avg_Income',
    'Annual Income_median': 'Median_Income',
    'Annual Income_std': 'Income_StdDev', 
    'Annual Income_count': 'Job_Count',
    'Employment_sum': 'Total_Employment'}
)

# Income premium vs national avg
national_avg_income = high_pay_data['Annual Income'].mean()
market_analysis['Income_Premium'] = ((market_analysis['Avg_Income'] - national_avg_income) / national_avg_income) * 100

# Market size buckets
job_count_terciles = np.percentile(market_analysis['Job_Count'], [33, 67])
market_analysis['Market_Size'] = pd.cut(
    market_analysis['Job_Count'],
    bins=[-np.inf, job_count_terciles[0], job_count_terciles[1], np.inf],
    labels=['Small', 'Medium', 'Large']
)

# Correlations
correlation_job_income = market_analysis['Job_Count'].corr(market_analysis['Avg_Income'])
correlation_employment_income = market_analysis['Total_Employment'].corr(market_analysis['Avg_Income'])

print("✅ Market analysis completed")
print(f"📊 Correlation (Job Count vs Income): {correlation_job_income:.3f}")
print(f"📊 Correlation (Total Employment vs Income): {correlation_employment_income:.3f}")

# Visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 16), dpi=300)

# 1. Market Size vs Average Income Scatter
market_colors = {'Small': '#E74C3C', 'Medium': '#F39C12', 'Large': '#27AE60'}
for market_size in ['Small', 'Medium', 'Large']:
    mask = market_analysis['Market_Size'] == market_size
    ax1.scatter(market_analysis[mask]['Job_Count'],
               market_analysis[mask]['Avg_Income'],
               c=market_colors[market_size], label=f'{market_size} Market',
               s=100, alpha=0.7, edgecolors='black')

# Trend line
z = np.polyfit(market_analysis['Job_Count'], market_analysis['Avg_Income'], 1)
p = np.poly1d(z)
ax1.plot(market_analysis['Job_Count'], p(market_analysis['Job_Count']),
         "r--", alpha=0.8, linewidth=2, label=f'Trend (r={correlation_job_income:.3f})')
ax1.set_xlabel('Job Count (High-Paying Positions)')
ax1.set_ylabel('Average Income ($)')
ax1.set_title('Market Size vs Average Income by State')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Annotate extremes
top_income_states = market_analysis.nlargest(3, 'Avg_Income')
large_markets = market_analysis.nlargest(3, 'Job_Count')
for idx, row in top_income_states.iterrows():
    ax1.annotate(idx, (row['Job_Count'], row['Avg_Income']),
                xytext=(10, 10), textcoords='offset points',
                fontsize=9, bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

# 2. Income Premium vs Market Size
market_size_stats = market_analysis.groupby('Market_Size').agg({
    'Income_Premium': ['mean', 'std', 'count'],
    'Avg_Income': 'mean'
}).round(2)
market_size_stats.columns = ['_'.join(col).strip() for col in market_size_stats.columns]
market_sizes = ['Small', 'Medium', 'Large']
premiums = [market_size_stats.loc[size, 'Income_Premium_mean'] for size in market_sizes]
premium_stds = [market_size_stats.loc[size, 'Income_Premium_std'] for size in market_sizes]
bars2 = ax2.bar(market_sizes, premiums,
               color=[market_colors[size] for size in market_sizes],
               alpha=0.8, edgecolor='black',
               yerr=premium_stds, capsize=5)
ax2.set_ylabel('Average Income Premium (%)')
ax2.set_title('Income Premium by Market Size (vs National Avg)')
ax2.axhline(y=0, color='red', linestyle='--', alpha=0.7, label='National Average')
ax2.legend()
for bar, value in zip(bars2, premiums):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(premiums)*0.01,
            f'{value:+.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

# 3. Top 15 by Market Size (Job Count)
top_15_markets = market_analysis.nlargest(15, 'Job_Count')
bars3 = ax3.bar(range(len(top_15_markets)), top_15_markets['Job_Count'],
               color=plt.cm.viridis(np.linspace(0, 1, len(top_15_markets))),
               alpha=0.8, edgecolor='black')
ax3.set_xticks(range(len(top_15_markets)))
ax3.set_xticklabels(top_15_markets.index, rotation=45, ha='right')
ax3.set_ylabel('Number of High-Paying Jobs')
ax3.set_title('Top 15 States by Market Size (High-Paying Job Count)')
for i, (bar, value) in enumerate(zip(bars3, top_15_markets['Job_Count'])):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(top_15_markets['Job_Count'])*0.01,
            f'{value:,}', ha='center', va='bottom', fontsize=9, rotation=90)

# 4. Market Efficiency (Income per Job)
market_analysis['Income_per_Job_Ratio'] = market_analysis['Avg_Income'] / market_analysis['Job_Count'] * 1000
top_efficient = market_analysis.nlargest(15, 'Income_per_Job_Ratio')
bars4 = ax4.barh(range(len(top_efficient)), top_efficient['Income_per_Job_Ratio'],
                color='#9B59B6', alpha=0.8, edgecolor='black')
ax4.set_yticks(range(len(top_efficient)))
ax4.set_yticklabels(top_efficient.index)
ax4.set_xlabel('Income Efficiency Ratio (Income × 1000 / Job Count)')
ax4.set_title('Top 15 Most Efficient Markets')
for i, (bar, value) in enumerate(zip(bars4, top_efficient['Income_per_Job_Ratio'])):
    ax4.text(bar.get_width() + max(top_efficient['Income_per_Job_Ratio'])*0.01,
            bar.get_y() + bar.get_height()/2, f'{value:.1f}', va='center', fontsize=10)

plt.suptitle('Question 8: Market Size vs Income Premium Analysis by State', fontsize=18, fontweight='bold', y=0.98)
plt.tight_layout()
os.makedirs('Images', exist_ok=True)
plt.savefig('Images/Market_Size_Income_Premium_Analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved figure: Images/Market_Size_Income_Premium_Analysis.png")

# Summary logs
print(f"\n📊 Market Analysis Key Insights:")
print(f"   • Correlation (Market Size ↔ Income): {correlation_job_income:.3f}")
print(f"   • Largest Market: {large_markets.index[0]} ({large_markets.iloc[0]['Job_Count']:,} jobs)")
if len(top_efficient) > 0:
    print(f"   • Most Efficient Market: {top_efficient.index[0]} (ratio: {top_efficient.iloc[0]['Income_per_Job_Ratio']:.1f})")

In [ ]:
# Q7 - Interpretation (auto-generated)
from IPython.display import Markdown, display
try:
    most_concentrated = max(occupation_geography.items(), key=lambda x: x[1]['Top_State_Percentage'])
    most_spread = max(occupation_geography.items(), key=lambda x: x[1]['Geographic_Spread'])
    display(Markdown(f"""
### Interpretation (Q7)
- Most geographically concentrated occupation: {most_concentrated[0]} ({most_concentrated[1]['Top_State_Percentage']:.1f}% in {most_concentrated[1]['Top_State']})
- Most geographically spread occupation: {most_spread[0]} ({most_spread[1]['Geographic_Spread']} states)
"""))
except Exception:
    pass

# Q8 - Interpretation (auto-generated)
try:
    display(Markdown(f"""
### Interpretation (Q8)
- Correlation (Job Count ↔ Income): {correlation_job_income:.3f}
- Largest market: {large_markets.index[0]} ({large_markets.iloc[0]['Job_Count']:,} jobs)
"""))
except Exception:
    pass